# Sheet 2.2: GPT from scratch and tokenization

**Roadmap topic 2.2** (Fanout 06.03). Watch first: Karpathy's *Let's build GPT*, *Let's build the GPT Tokenizer* and *Let's reproduce GPT-2 (124M)*. Code to study: `karpathy/build-nanogpt`, `karpathy/minbpe`, Raschka's *LLMs-from-scratch* ch. 2-5.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Checks run on CPU without downloading anything (they build randomly initialised Hugging Face models from a config). The training run in Exercise 5 needs a Colab T4.

`pip install transformers` if your runtime lacks it.

In [ ]:
import math, time, collections, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

## Exercise 1: Byte-level BPE

- `train_bpe(text, vocab_size)`: start from the 256 byte tokens and repeatedly merge the most frequent adjacent pair (ties broken by the pair that appears **first** in the current token sequence). Return `merges`, a dict `{(a, b): new_id}` in creation order, ids starting at 256.
- `encode(text, merges)`: UTF-8 bytes, then apply merges in **rank order** (always merge the pair with the lowest new id present), like GPT-2.
- `decode(ids, merges)`: back to a string (`errors="replace"`).

In [ ]:
def train_bpe(text, vocab_size):
    # YOUR CODE
    raise NotImplementedError
def encode(text, merges):
    # YOUR CODE
    raise NotImplementedError
def decode(ids, merges):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    text = ("the cat sat on the mat. the dog sat on the log. " * 20) + "naïve café, 日本語のテキスト, emoji 🙂🙂 "
    m = train_bpe(text, 300)
    assert list(m.values()) == list(range(256, 256 + len(m))), "ids must be 256, 257, ... in creation order"
    first = collections.Counter(zip(text.encode(), text.encode()[1:])).most_common(1)[0][0]
    assert next(iter(m)) == first, "first merge must be the most frequent byte pair"
    for s in (text, "the theme of the thesis", "日本🙂 the mat", ""):
        assert decode(encode(s, m), m) == s, f"round trip failed on {s!r}"
    ids = encode(text, m)
    assert len(ids) < 0.35 * len(text.encode()), f"poor compression: {len(ids)} tokens for {len(text.encode())} bytes"
    assert encode("the", m) == encode("the", m) and len(encode("the ", m)) <= 2
    print(f"PASS  compression {len(text.encode()) / len(ids):.2f} bytes/token")
_check1()

### On Colab: match GPT-2's tokenizer
`pip install tiktoken regex`. GPT-2 first splits text with a regex (`enc._pat_str`) and only merges inside each chunk. Load `tiktoken.get_encoding("gpt2")`, rebuild its `merges` dict from `enc._mergeable_ranks` (as in minbpe's `gpt4.py`), add the regex split to your `encode`, and verify you produce **identical ids** to `enc.encode` on 1 MB of mixed text (Wikipedia + code + emoji).

In [ ]:
# Colab: tiktoken parity

## Exercise 2: GPT-2 in ~100 lines, and load real weights into it

Write `GPT(cfg)` with the GPT-2 architecture: token + learned position embeddings, `n_layer` pre-LN blocks (LayerNorm → causal self-attention → residual, LayerNorm → MLP with GELU(tanh) and 4× width → residual), final LayerNorm, LM head **tied** to the token embedding. `forward(idx)` returns logits `(B, T, vocab)`.

Then `load_hf_gpt2(model, hf_state_dict)` copies weights from a Hugging Face `GPT2LMHeadModel` state dict. Hugging Face uses `Conv1D` layers whose weights are stored transposed relative to `nn.Linear`.

In [ ]:
from dataclasses import dataclass
@dataclass
class Cfg:
    vocab_size: int = 50257; n_ctx: int = 1024; n_layer: int = 12; n_head: int = 12; n_embd: int = 768

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # YOUR CODE
        raise NotImplementedError
    def forward(self, idx):
        # YOUR CODE
        raise NotImplementedError

def load_hf_gpt2(model, sd):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
@torch.no_grad()
def _check2():
    from transformers import GPT2Config, GPT2LMHeadModel
    torch.manual_seed(0)
    hc = GPT2Config(vocab_size=1000, n_positions=64, n_layer=3, n_head=4, n_embd=128, attn_implementation="eager", bos_token_id=0, eos_token_id=0)
    hf = GPT2LMHeadModel(hc).eval()
    for p in hf.parameters(): p.add_(torch.randn_like(p) * 0.02)       # make biases/LN non-trivial
    hf.lm_head.weight = hf.transformer.wte.weight
    mine = GPT(Cfg(vocab_size=1000, n_ctx=64, n_layer=3, n_head=4, n_embd=128)).eval()
    load_hf_gpt2(mine, hf.state_dict())
    idx = torch.randint(0, 1000, (2, 40))
    a, b = mine(idx), hf(idx).logits
    assert a.shape == b.shape, f"shape {a.shape} vs {b.shape}"
    assert torch.allclose(a, b, atol=1e-4), f"logits differ, max {(a - b).abs().max():.2e}"
    n_params = sum(p.numel() for p in GPT(Cfg()).parameters())
    assert abs(n_params - 124_439_808) < 1000, f"full-size param count {n_params:,} (should be about 124.4M with tying)"
    print(f"PASS  logits match; full GPT-2 small has {n_params:,} parameters")
_check2()

On Colab, repeat with `GPT2LMHeadModel.from_pretrained("gpt2")` and the real config, and sample 50 tokens from "The meaning of life is" with your model.

## Exercise 3: Gradient accumulation that is actually correct

You train a causal LM with padding, so micro-batches contain **different numbers of real tokens**. The loss you want is the mean over **all real tokens in the global batch**. Implement `accumulate(model, micro_batches)` that runs forward/backward on each micro-batch (each a tuple `(idx, targets, mask)`), leaves in `.grad` exactly the gradient of the global-batch loss, and returns that loss.

The naive version (average of per-micro-batch means) is a real bug that shipped in popular trainers in 2024.

In [ ]:
def accumulate(model, micro_batches):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    torch.manual_seed(0); torch.set_default_dtype(torch.float64)
    model = GPT(Cfg(vocab_size=50, n_ctx=16, n_layer=2, n_head=2, n_embd=32))
    idx = torch.randint(0, 50, (8, 16)); tgt = torch.randint(0, 50, (8, 16))
    lengths = torch.tensor([16, 3, 9, 1, 16, 12, 2, 7]); mask = (torch.arange(16)[None] < lengths[:, None]).double()
    model.zero_grad()
    full = (F.cross_entropy(model(idx).flatten(0, 1), tgt.flatten(), reduction="none") * mask.flatten()).sum() / mask.sum()
    full.backward(); ref = [p.grad.clone() for p in model.parameters()]
    mbs = [(idx[i:i + 2], tgt[i:i + 2], mask[i:i + 2]) for i in range(0, 8, 2)]
    loss = accumulate(model, mbs)
    assert abs(loss - full.item()) < 1e-10, f"loss {loss} vs {full.item()}"
    for p, r in zip(model.parameters(), ref):
        assert torch.allclose(p.grad, r, atol=1e-12), "accumulated gradient differs from full-batch gradient"
    torch.set_default_dtype(torch.float32)
    print("PASS")
_check3()

## Exercise 4: FLOPs and MFU

- `train_flops_per_token(n_params, n_layer, T, C)`: the standard estimate `6N + 12·n_layer·T·C` (Kaplan / PaLM appendix B), where N excludes embedding parameters.
- `mfu(tokens_per_sec, n_params, n_layer, T, C, peak_flops)`.
- `non_embedding_params(model)` for your `GPT` (exclude `wte` and `wpe`).

In [ ]:
def non_embedding_params(model): raise NotImplementedError
def train_flops_per_token(n_params, n_layer, T, C): raise NotImplementedError
def mfu(tokens_per_sec, n_params, n_layer, T, C, peak_flops): raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    m = GPT(Cfg()); N = non_embedding_params(m)
    assert abs(N - 85_056_000) < 50_000, f"non-embedding params {N:,}"
    f = train_flops_per_token(N, 12, 1024, 768)
    assert abs(f - (6 * N + 12 * 12 * 1024 * 768)) < 1
    u = mfu(20_000, N, 12, 1024, 768, 65e12)
    print(f"PASS  GPT-2 small: {f/1e9:.2f} GFLOP/token; 20k tok/s on a T4 (65 TFLOPS fp16) = {u:.1%} MFU")
_check4()

## Exercise 5 (Colab T4): train it
Train a ~20M-parameter version of your GPT (`n_layer=6, n_head=6, n_embd=384`, context 256) on **TinyStories** (`datasets.load_dataset("roneneldan/TinyStories")`) with your BPE or the GPT-2 tokenizer, AdamW (β₂ = 0.95), warmup + cosine, `torch.autocast(dtype=torch.float16)` + `GradScaler` (T4 has no bf16), and correct gradient accumulation.

**Targets:** validation loss < 1.6 nats/token with the GPT-2 tokenizer within 2 hours; report tokens/sec and MFU; show 5 samples at the end that are coherent stories. Plot train/val loss.

In [ ]:
# Colab: TinyStories training run

## Checkpoint
Explain every line of your model and training script to someone else, including: why weight tying, why pre-norm, what loss a uniform predictor gets for your vocabulary (compute it), and why the masked-token loss in Exercise 3 must divide by the global token count.

---
### Hints
- Uniform predictor loss = ln(vocab_size): ln(50257) ≈ 10.82 nats.